In [2]:
from sly import Lexer, Parser

In [3]:
class CalcLexer(Lexer):
    tokens = {
        CONTRACT,
        ID,
        LBRACE,
        RBRACE,
        LPAREN,
        RPAREN,
        IF,
        ELSE,
        WHILE,
        FUNC,
        SEMICOLON,
        RARROW,
        RETURN,
        PLUS,
        MINUS,
        TIMES,
        DIVIDE,
        ASSIGN,
        NUMBER,
    }

    # Tokens
    LBRACE = r"\{"
    LPAREN = r"\("
    RBRACE = r"\}"
    RPAREN = r"\)"
    SEMICOLON = r"\;"
    RARROW = r"->"
    PLUS = r"\+"
    MINUS = r"-"
    TIMES = r"\*"
    DIVIDE = r"/"
    ASSIGN = r"="
    NUMBER = r"\d+"

    ID = r"[a-zA-Z_][a-zA-Z0-9_]*"

    # Special cases
    ID["impl"] = CONTRACT
    ID["if"] = IF
    ID["else"] = ELSE
    ID["while"] = WHILE
    ID["fn"] = FUNC
    ID["return"] = RETURN

    # Ignored pattern
    ignore = ' \t'
    ignore_newline = r"\n+"

    # Extra action for newlines
    def ignore_newline(self, t):
        self.lineno += t.value.count("\n")

    def error(self, t):
        print("Illegal character '%s'" % t.value[0])
        self.index += 1


class CalcParser(Parser):
    tokens = CalcLexer.tokens

    precedence = (
        ("left", PLUS, MINUS),
        ("left", TIMES, DIVIDE),
        #("right", UMINUS),
    )

    def __init__(self):
        self.names = {}

    """
    sourceInit: contractDefinition | #constantDeclaration# | #functionDefinition#

    contractDefintion: CONTRACT ID LBRACE contractBody RBRACE
    
    contractBody: functionDefinition | #globalVariableDeclaration# | empty

    functionDefinition: FUNC ID LPAREN ID RPAREN RARROW LPAREN ID RPAREN LBRACE functionCode RBRACE SEMICOLON

    functionCode: RETURN expression SEMICOLON | ID ASSIGN expression SEMICOLON

    expression : expression PLUS expression
     | expression MINUS expression
     | expression TIMES expression
     | expression DIVIDE expression
     | LPAREN expression RPAREN
     | NUMBER
    """

    @_("contractDefinition")
    def sourceInit(self, p):
        return ("CONTRACT DEF", [], p[0])

    @_("CONTRACT ID LBRACE contractBody RBRACE")
    def contractDefinition(self, p):
        return ("CONTRACT", p.ID, p.contractBody)

    @_("functionDefinition contractBody", "empty")
    def contractBody(self, p):
        if hasattr(p, "functionDefinition"):
            return ("FUNCTION DEF", p.functionDefinition, p.contractBody)
        return ("EMPTY", None, None)

    @_(
        "FUNC ID LPAREN ID RPAREN RARROW LPAREN ID RPAREN LBRACE functionCode RBRACE SEMICOLON",
        "FUNC ID LPAREN ID RPAREN LBRACE functionCode RBRACE SEMICOLON",
    )
    def functionDefinition(self, p):
        if hasattr(p, "functionCode"):
            functor = p.ID0 + p[2] + p.ID1 + p[4]
            return ("FUNCTION", functor, p.functionCode)

    @_("RETURN expression SEMICOLON")
    def functionCode(self, p):
        return ("RETURN", p.expression, None)

    @_("ID ASSIGN expression SEMICOLON")
    def functionCode(self, p):
        return ("ASSIGN", p.expression, None)
        

    @_('expression PLUS expression',
    'expression MINUS expression',
    'expression TIMES expression',
    'expression DIVIDE expression')
    def expression(self, p):
        return ('binaryOperation', p[1], p.expression0, p.expression1)

    @_('LPAREN expression RPAREN')
    def expression(self, p):
        return ('groupExpression',p.expression)

    @_('NUMBER')
    def expression(self, p):
        return ('numberExpression', p.NUMBER)

    @_('ID')
    def expression(self, p):
        return ('namedExpression', p.ID)

    @_("")
    def empty(self, p):
        return ("EMPTY", None, None)


In [4]:
lexer = CalcLexer()
parser = CalcParser()

text = '''
impl contrateto { 
   fn algoif(roberto) -> (uint256) {
      return roberto + 1;
   };
   fn algoels(alejandro) {
      algo = alejandro + 3; 
   };
   fn algowhile(roxana) -> (uint256) {
      roxana = 2;
   };
}
'''

for a in lexer.tokenize(text):
   print(a)

tree = parser.parse(lexer.tokenize(text))
tree

Token(type='CONTRACT', value='impl', lineno=2, index=1)
Token(type='ID', value='contrateto', lineno=2, index=6)
Token(type='LBRACE', value='{', lineno=2, index=17)
Token(type='FUNC', value='fn', lineno=3, index=23)
Token(type='ID', value='algoif', lineno=3, index=26)
Token(type='LPAREN', value='(', lineno=3, index=32)
Token(type='ID', value='roberto', lineno=3, index=33)
Token(type='RPAREN', value=')', lineno=3, index=40)
Token(type='RARROW', value='->', lineno=3, index=42)
Token(type='LPAREN', value='(', lineno=3, index=45)
Token(type='ID', value='uint256', lineno=3, index=46)
Token(type='RPAREN', value=')', lineno=3, index=53)
Token(type='LBRACE', value='{', lineno=3, index=55)
Token(type='RETURN', value='return', lineno=4, index=63)
Token(type='ID', value='roberto', lineno=4, index=70)
Token(type='PLUS', value='+', lineno=4, index=78)
Token(type='NUMBER', value='1', lineno=4, index=80)
Token(type='SEMICOLON', value=';', lineno=4, index=81)
Token(type='RBRACE', value='}', lineno=5, i

('CONTRACT DEF',
 [],
 ('CONTRACT',
  'contrateto',
  ('FUNCTION DEF',
   ('FUNCTION',
    'algoif(roberto)',
    ('RETURN',
     ('binaryOperation',
      '+',
      ('namedExpression', 'roberto'),
      ('numberExpression', '1')),
     None)),
   ('FUNCTION DEF',
    ('FUNCTION',
     'algoels(alejandro)',
     ('ASSIGN',
      ('binaryOperation',
       '+',
       ('namedExpression', 'alejandro'),
       ('numberExpression', '3')),
      None)),
    ('FUNCTION DEF',
     ('FUNCTION',
      'algowhile(roxana)',
      ('ASSIGN', ('numberExpression', '2'), None)),
     ('EMPTY', None, None))))))

In [5]:
from sha3 import keccak_256


class Contract:

    def __init__(self, functions=[]) -> None:
        # TODO Create Code from function selectors
        code = '    0x00 calldataload 0xE0 shr \n'
        jumps = []
        for fn in functions:
            func = bytes(fn.selector, encoding='utf8')
            sha3_hash = keccak_256(func).hexdigest()
            funcSelector = "0x"+sha3_hash[:8]
            code += f'    dup1 {funcSelector} eq {fn.name} jumpi\n'
            jumps.append(fn.name)

        code += '\n    0x00 0x00 revert\n'
        
        for jump in jumps:
            code += '    ' + jump + ':\n' + '        ' + jump.upper() + '()' + '\n'

        self.main = Function('MAIN', '', 0, 0, code)
        self.functions = functions

    def __repr__(self) -> str:

        main =  repr(self.main)
        funcs = ''
        for func in self.functions:
            funcs = funcs + repr(func) + '\n'

        return main + '\n' + funcs
        

class Interface:

    def __init__(self, selector, returns=None, modifiers=None) -> None:
        self.selector = selector
        self.returns = returns
        self.modifiers = modifiers

    def __repr__(self) -> str:
        define = f'#define function {self.selector} '
        define += ' '.join(self.modifiers)
        if self.returns is not None:
            define += 'returns ({self.returns})'
        return define

class Function:

    def __init__(self, selector, returns, n_takes, n_returns, code) -> None:
        self.selector = selector
        self.returns = returns
        self.n_takes = n_takes
        self.n_returns = n_returns
        self.code = code

    @property
    def name(self):
        return self.selector.split('(')[0]

    def functor(self):
        # TODO FUNCTOR SELECTOR
        return self.name

    def __repr__(self) -> str:
        return f'#define macro {self.name.upper()}() = takes ({self.n_takes}) returns ({self.n_returns})' + ' { \n' + str(self.code) + ' \n}'


class Code:

    def __init__(self, instructions=None) -> None:
        if instructions is None:
            instructions = []
        self.instructions = instructions

    def __repr__(self) -> str:
        insts = ''
        for inst in self.instructions:
            insts = insts + '\n' + repr(inst)

        



In [6]:
class BasicExecute: 
	
	def __init__(self):#, env): 
		#self.env = env 
		'''result = self.walkTree(tree) 
		if result is not None and isinstance(result, int): 
			print(result) 
		if isinstance(result, str) and result[0] == '"': 
			print(result) 
		'''

	def run(self, tree):
		result = self.walkTree(tree) 
		return result

	def walkTree(self, node): 

		if isinstance(node, int): 
			return node 
		if isinstance(node, str): 
			return node 

		if node is None: 
			return None

		if node[0] == 'CONTRACT DEF':
			print('CONTRACT DEF')
			result = self.walkTree(node[2])
		elif node[0] == 'CONTRACT':
			print('CONTRACT')
			name = node[1]
			functions = self.walkTree(node[2])
			result = Contract(functions)
		elif node[0] == 'FUNCTION DEF':
			print('FUNCTION DEF')
			result0 = self.walkTree(node[1])
			result = self.walkTree(node[2])
			if result is not None:
				result.append(result0)
			else:
				return [result0]
		elif node[0] == 'FUNCTION':
			print('FUNCTION')
			selector = node[1]
			returns = None
			code = self.walkTree(node[2])
			result = Function(selector, returns, 0, 0, code)
		elif node[0] == 'ASSIGN':
			print('ASSIGN')
			result = self.walkTree(node[1])
		elif node[0] == 'RETURN':
			print('RETURN')
			result = self.walkTree(node[1])
		elif node[0] == 'binaryOperation':
			print('BINARY')
			if node[1] == '+':
				lnode = self.walkTree(node[2])
				rnode = self.walkTree(node[3])
				print(lnode, rnode)
				result = lnode + rnode
		elif node[0] == 'namedExpression':
			print('NAMED')
			# TODO from var to value using table
			result = 5
		elif node[0] == 'numberExpression':
			print('NUMBER')
			result = int(node[1])
		elif node[0] == 'EMPTY':
			print('EMPTY')
			return None
		else:
			print(node)
			raise Exception('Error', node)

		return result

		'''if node[0] == 'num': 
			return node[1] 

		if node[0] == 'str': 
			return node[1] 

		if node[0] == 'add': 
			return self.walkTree(node[1]) + self.walkTree(node[2]) 
		elif node[0] == 'sub': 
			return self.walkTree(node[1]) - self.walkTree(node[2]) 
		elif node[0] == 'mul': 
			return self.walkTree(node[1]) * self.walkTree(node[2]) 
		elif node[0] == 'div': 
			return self.walkTree(node[1]) / self.walkTree(node[2]) 

		if node[0] == 'var_assign': 
			self.env[node[1]] = self.walkTree(node[2]) 
			return node[1] 

		if node[0] == 'var': 
			try: 
				return self.env[node[1]] 
			except LookupError: 
				print("Undefined variable '"+node[1]+"' found!") 
				return 0'''

In [7]:
aca = BasicExecute().run(tree)

CONTRACT DEF
CONTRACT
FUNCTION DEF
FUNCTION
RETURN
BINARY
NAMED
NUMBER
5 1
FUNCTION DEF
FUNCTION
ASSIGN
BINARY
NAMED
NUMBER
5 3
FUNCTION DEF
FUNCTION
ASSIGN
NUMBER
EMPTY


In [8]:
aca

#define macro MAIN() = takes (0) returns (0) { 
    0x00 calldataload 0xE0 shr 
    dup1 0x6dd6162b eq algowhile jumpi
    dup1 0xe81e5bfe eq algoels jumpi
    dup1 0x70e0287a eq algoif jumpi

    0x00 0x00 revert
    algowhile:
        ALGOWHILE()
    algoels:
        ALGOELS()
    algoif:
        ALGOIF()
 
}
#define macro ALGOWHILE() = takes (0) returns (0) { 
2 
}
#define macro ALGOELS() = takes (0) returns (0) { 
8 
}
#define macro ALGOIF() = takes (0) returns (0) { 
6 
}

In [27]:
'''
/* Interface */
#define function setValue(uint256) nonpayable returns ()
#define function getValue() view returns (uint256)

/* Storage Slots */
#define constant VALUE_LOCATION = FREE_STORAGE_POINTER()

/* Methods */
#define macro SET_VALUE() = takes (0) returns (0) {
    0x04 calldataload   // [value]
    [VALUE_LOCATION]    // [ptr, value]
    sstore              // []
}

#define macro GET_VALUE() = takes (0) returns (0) {
    // Load value from storage.
    [VALUE_LOCATION]   // [ptr]
    sload                // [value]

    // Store value in memory.
    0x00 mstore

    // Return value
    0x20 0x00 return
}

#define macro MAIN() = takes (0) returns (0) {
    // Identify which function is being called.
    0x00 calldataload 0xE0 shr
    dup1 0x55241077 eq set jumpi
    dup1 0x20965255 eq get jumpi

    set:
        SET_VALUE()
    get:
        GET_VALUE()
}
'''

'\n/* Interface */\n#define function setValue(uint256) nonpayable returns ()\n#define function getValue() view returns (uint256)\n\n/* Storage Slots */\n#define constant VALUE_LOCATION = FREE_STORAGE_POINTER()\n\n/* Methods */\n#define macro SET_VALUE() = takes (0) returns (0) {\n    0x04 calldataload   // [value]\n    [VALUE_LOCATION]    // [ptr, value]\n    sstore              // []\n}\n\n#define macro GET_VALUE() = takes (0) returns (0) {\n    // Load value from storage.\n    [VALUE_LOCATION]   // [ptr]\n    sload                // [value]\n\n    // Store value in memory.\n    0x00 mstore\n\n    // Return value\n    0x20 0x00 return\n}\n\n#define macro MAIN() = takes (0) returns (0) {\n    // Identify which function is being called.\n    0x00 calldataload 0xE0 shr\n    dup1 0x55241077 eq set jumpi\n    dup1 0x20965255 eq get jumpi\n\n    set:\n        SET_VALUE()\n    get:\n        GET_VALUE()\n}\n'